# Tsena — Demo & Test Notebook

Run cells top-to-bottom. **Sections 1–2 need no Ollama** (pure Python).  
From section 3 onward Ollama must be running with at least `mistral:7b` installed.

```
ollama pull mistral:7b
ollama serve          # if not already running
```

| Section | Needs Ollama? |
|---|---|
| 1 · Setup | No |
| 2 · Parser unit tests | No |
| 3 · Model discovery | No (just reads `ollama list`) |
| 4 · Task breakdown | Yes |
| 5 · Per-task token measurement | Yes |
| 6 · Cost estimation | Yes (uses section 5 results) |
| 7 · Full non-interactive pipeline | Yes |
| 8 · CLI tool demo | Yes |
| 9 · HTTP microservice | Yes + service on port 5001 |

## 1 · Setup

In [9]:
import sys
import os
import json
import subprocess
import importlib.util
import time
import urllib.request

# Walk up from CWD until we find the project root (the dir that contains src/)
ROOT = os.path.abspath('.')
while ROOT != os.path.dirname(ROOT):
    if os.path.isdir(os.path.join(ROOT, 'src')):
        break
    ROOT = os.path.dirname(ROOT)

SRC   = os.path.join(ROOT, 'src')
TOOLS = os.path.join(SRC, 'tools')

for p in (SRC, TOOLS):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Project root : {ROOT}')
print(f'Python       : {sys.version.split()[0]}')
print()

r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
if r.returncode == 0:
    print('Ollama is running. Installed models:')
    print(r.stdout.strip())
else:
    print('WARNING: Ollama is not running or not installed.')
    print('Download: https://ollama.com/download  |  start: ollama serve')

Project root : /Users/viktor/PycharmProjects/tsena
Python       : 3.14.3

Ollama is running. Installed models:
NAME                    ID              SIZE      MODIFIED    
mistral:7b              6577803aa9a0    4.4 GB    5 days ago     
qwen2.5:3b              357c53fb659c    1.9 GB    4 weeks ago    
mistral:latest          6577803aa9a0    4.4 GB    4 weeks ago    
qwen3-embedding:0.6b    ac6da0dfba84    639 MB    4 weeks ago


## 2 · Parser Unit Tests

Tests the plain-language task parser (`_parse_plain_language_tasks`) without any LLM call.  
Covers numbered lists, dash lists, and bullet-point lists.

In [10]:
from task_breaker import _parse_plain_language_tasks

NUMBERED = """
1. Set up project structure
Create the directory layout and install dependencies.
Difficulty: 1
Estimated: 1.5 hours

2. Implement database schema
Design tables for users and posts.
Difficulty: 3
Estimated: 4 hours
Depends on: Set up project structure

3. Build API endpoints
Create REST endpoints for CRUD operations.
Difficulty: 4
Estimated: 8 hours
Depends on: Implement database schema
"""

DASHES = """
- Task One
Simple clean-up tasks.
difficulty: 2
estimated: 2 hours

- Task Two
Build the main feature.
difficulty: 4
estimated: 6 hours
"""

BULLETS = """
• Authentication System
Implement user login and registration.
Difficulty: 3
Estimated: 5 hours

• Payment Integration
Integrate payment gateway.
Difficulty: 4
Estimated: 6 hours
depends on: Authentication System
"""

def run_parser_test(name, text, expected_count, checks=None):
    tasks = _parse_plain_language_tasks(text)
    ok = len(tasks) == expected_count
    if checks:
        for check_fn, msg in checks:
            if not check_fn(tasks):
                print(f'  FAIL  {name}: {msg}')
                return False
    if not ok:
        print(f'  FAIL  {name}: expected {expected_count} tasks, got {len(tasks)}')
        return False
    for t in tasks:
        print(f'  + {t.title} (diff={t.difficulty}, {t.estimated_hours}h)')
    print(f'  PASS  {name} — {len(tasks)} tasks parsed')
    return True

results = []
print('=== Parser Tests ===')
print()

print('Test 1: numbered list')
results.append(run_parser_test(
    'numbered list', NUMBERED, 3,
    checks=[
        (lambda ts: ts[0].difficulty == 1, 'task[0].difficulty should be 1'),
        (lambda ts: ts[1].estimated_hours == 4.0, 'task[1].hours should be 4'),
        (lambda ts: len(ts[2].dependencies) > 0, 'task[2] should have deps'),
    ]
))
print()

print('Test 2: dash list')
results.append(run_parser_test('dash list', DASHES, 2))
print()

print('Test 3: bullet list')
results.append(run_parser_test(
    'bullet list', BULLETS, 2,
    checks=[
        (lambda ts: len(ts[1].dependencies) > 0, 'task[1] should have deps'),
    ]
))
print()

passed = sum(results)
print(f'Result: {passed}/{len(results)} tests passed', '✓' if passed == len(results) else '✗')

=== Parser Tests ===

Test 1: numbered list
  + Set up project structure (diff=1, 1.5h)
  + Implement database schema (diff=3, 4.0h)
  + Build API endpoints (diff=4, 8.0h)
  PASS  numbered list — 3 tasks parsed

Test 2: dash list
  + Task One (diff=2, 2.0h)
  + Task Two (diff=4, 6.0h)
  PASS  dash list — 2 tasks parsed

Test 3: bullet list
  + Authentication System (diff=3, 5.0h)
  + Payment Integration (diff=4, 6.0h)
  PASS  bullet list — 2 tasks parsed

Result: 3/3 tests passed ✓


## 3 · Model Discovery

Auto-detects all generative (non-embedding) models from `ollama list`, deduplicating aliases by model ID.

In [11]:
from task_breaker import get_smallest_generative_model, get_largest_generative_model

SMALLEST = get_smallest_generative_model()
LARGEST  = get_largest_generative_model()

print(f'Smallest generative model : {SMALLEST}')
print(f'Largest generative model  : {LARGEST}')
print()

# Show the full deduplicated list
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
seen_ids = set()
models = []
for line in r.stdout.strip().split('\n')[1:]:
    parts = line.split()
    if len(parts) < 4:
        continue
    name, mid = parts[0], parts[1]
    if 'embed' in name.lower() or mid in seen_ids:
        continue
    seen_ids.add(mid)
    size = f'{parts[2]} {parts[3]}'
    models.append((name, size))

print('Generative models available (deduplicated):')
for name, size in models:
    tag = '  <- will be used for measurement' if name == LARGEST else ''
    print(f'  {name:30s} {size}{tag}')

Smallest generative model : qwen2.5:3b
Largest generative model  : mistral:7b

Generative models available (deduplicated):
  mistral:7b                     4.4 GB  <- will be used for measurement
  qwen2.5:3b                     1.9 GB


## 4 · Task Breakdown

One LLM call that decomposes a project description into numbered subtasks.  
Token usage is captured via the Ollama REST API streaming endpoint.

In [12]:
from task_breaker import break_down_task, format_task_breakdown

PROJECT = 'Build a REST API with JWT authentication and a PostgreSQL database'

print(f'Model  : {LARGEST}')
print(f'Project: {PROJECT}')
print()

breakdown = break_down_task(PROJECT, model=LARGEST, temperature=0.3, max_retries=3)
print(format_task_breakdown(breakdown))

print()
if breakdown.total_prompt_tokens:
    total = breakdown.total_prompt_tokens + breakdown.total_response_tokens
    print(f'Breakdown token usage: prompt={breakdown.total_prompt_tokens},'
          f' response={breakdown.total_response_tokens}, total={total}')
print(f'Tasks parsed: {len(breakdown.tasks)}')

Model  : mistral:7b
Project: Build a REST API with JWT authentication and a PostgreSQL database

  [estimator] progress 1/1 at 420 tokens → projected ~420 total → stopping stream at 420 (50%)
  [estimator] reached 50% threshold (420 tokens), closing stream
  Token usage: prompt=0, response=420, total=420
Task Breakdown: Build a REST API with JWT authentication and a PostgreSQL database
Total Estimated Hours: 22.0
--------------------------------------------------------------------------------

1. Task Title One: Set up development environment
   Difficulty: 🔴🔴🔴⚪⚪ (3/5)
   Estimated: 1.0 hours
   Description: 

2. Install required dependencies for Node.js, PostgreSQL, and JWT libraries.
   Difficulty: 🔴🔴⚪⚪⚪ (2/5)
   Estimated: 1.0 hours
   Description: 
   Dependencies: none

3. Task Title Two: Configure PostgreSQL database connection
   Difficulty: 🔴🔴🔴⚪⚪ (3/5)
   Estimated: 1.0 hours
   Description: 

4. Create a new PostgreSQL database and define the necessary tables for the REST API.

## 5 · Per-Task Token Measurement

Each task is sent to the LLM with a system prompt asking it to write real implementation code.  
The stream is cut at **50%** of the extrapolated total once a `[PROGRESS: 1/10]` marker appears.

> `MEASURE_N` limits how many tasks to run. Increase to `len(breakdown.tasks)` for a full measurement.

In [13]:
from task_breaker import run_task_implementation

MEASURE_N = 3  # increase to len(breakdown.tasks) for full measurement

tasks_to_measure = breakdown.tasks[:MEASURE_N]
print(f'Measuring {len(tasks_to_measure)} of {len(breakdown.tasks)} tasks with: {LARGEST}')
print('(stream stops at 50% of projected total when a progress marker is detected)')
print('-' * 70)

measurements = []
for i, task in enumerate(tasks_to_measure, 1):
    print(f'[{i}/{len(tasks_to_measure)}] {task.title}')
    m = run_task_implementation(task, model=LARGEST)
    measurements.append(m)
    print(f'  prompt={m.prompt_tokens}, response={m.response_tokens}, total={m.total_tokens}')
    print()

avg_tokens = sum(m.total_tokens for m in measurements) / len(measurements)
print('-' * 70)
print(f'Average tokens per task (measured): {avg_tokens:.0f}')

Measuring 3 of 14 tasks with: mistral:7b
(stream stops at 50% of projected total when a progress marker is detected)
----------------------------------------------------------------------
[1/3] Task Title One: Set up development environment
  [estimator] progress 1/1 at 43 tokens → projected ~43 total → stopping stream at 43 (50%)
  [estimator] reached 50% threshold (43 tokens), closing stream
  prompt=0, response=43, total=43

[2/3] Install required dependencies for Node.js, PostgreSQL, and JWT libraries.
  [estimator] progress 1/1 at 89 tokens → projected ~89 total → stopping stream at 89 (50%)
  [estimator] reached 50% threshold (89 tokens), closing stream
  prompt=0, response=89, total=89

[3/3] Task Title Two: Configure PostgreSQL database connection
  [estimator] progress 1/1 at 146 tokens → projected ~146 total → stopping stream at 146 (50%)
  [estimator] reached 50% threshold (146 tokens), closing stream
  prompt=0, response=146, total=146

-------------------------------------

## 6 · Cost Estimation

Uses the real measured token average to project cost across the cloud-model catalogue (`LLM_MODELS`).  
The selector targets ~50% of the budget and respects task difficulty ranges.

In [14]:
# Load expense_estimator from the 'expense estimator' directory (space in name, use importlib)
_spec = importlib.util.spec_from_file_location(
    'expense_estimator',
    os.path.join(ROOT, 'src', 'tools', 'expense_estimator.py')
)
_ee = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ee)
LLM_MODELS       = _ee.LLM_MODELS
select_best_model = _ee.select_best_model

BUDGET = 5.0
avg_difficulty = sum(t.difficulty for t in breakdown.tasks) / len(breakdown.tasks)

print(f'Budget         : ${BUDGET}')
print(f'Avg difficulty : {avg_difficulty:.1f} / 5')
print(f'Avg tokens/task: {avg_tokens:.0f}  (from real measurement)')
print()

model_name, model_info, est_cost, reason = select_best_model(
    BUDGET, avg_difficulty, actual_tokens_per_task=avg_tokens
)

print()
print('=== RECOMMENDATION ===')
print(f'Model         : {model_info["display_name"]}')
print(f'Estimated cost: ${est_cost:.4f}')
print(f'Remaining     : ${BUDGET - est_cost:.2f}')
print(f'Reason        : {reason}')

Budget         : $5.0
Avg difficulty : 3.0 / 5
Avg tokens/task: 93  (from real measurement)

SELECTING OPTIMAL LLM MODEL (ML-Aware, Targeting 50% Budget)

  Task Difficulty Average: 3.0/5 (ML Project)
  Available Budget: $5.00
  Target Spending (50%): $2.50
  Token count source: actual measured (92.66666666666667 tokens/task)

Available models (sorted by 50% target + ML capability):
  1. Llama 2 70B (Premium - ML Ready)
     Cost: $0.0074 (0.1% of budget)
     Tokens/task: 92.66666666666667 (actual measured)
  2. Qwen 2.5 14B (Advanced)
     Cost: $0.0046 (0.1% of budget)
     Tokens/task: 92.66666666666667 (actual measured)
  3. Qwen 2.5 7B (Balanced)
     Cost: $0.0028 (0.1% of budget)
     Tokens/task: 92.66666666666667 (actual measured)
  4. Qwen 2.5 3B (Fast)
     Cost: $0.0009 (0.0% of budget)
     Tokens/task: 92.66666666666667 (actual measured)

  Selected: Llama 2 70B (Premium - ML Ready)
   Estimated Cost: $0.0074
   Budget Utilization: 0.1% (Target: 50%)


=== RECOMMENDATION

## 7 · Full Non-Interactive Pipeline

Runs the complete expense-estimator flow programmatically on a different project.  
Limits per-task measurement to 3 tasks; scale `MEASURE_N` up as needed.

In [15]:
from task_breaker import TaskBreakdown, measure_tasks

PIPELINE_BUDGET  = 10.0
PIPELINE_MEASURE = 3
PIPELINE_PROJECT = (
    'Build a mobile expense tracker with user login, '
    'add/edit/delete expenses, category tags, monthly summary charts, and CSV export'
)

print(f'Project : {PIPELINE_PROJECT}')
print(f'Budget  : ${PIPELINE_BUDGET}')
print(f'Model   : {LARGEST}')
print('=' * 70)

# Step 1: Task breakdown
print('\n--- Task Breakdown ---')
full_bd = break_down_task(PIPELINE_PROJECT, model=LARGEST, temperature=0.3, max_retries=3)
print(format_task_breakdown(full_bd))

# Step 2: Measure first N tasks
print(f'\n--- Per-Task Measurement (first {PIPELINE_MEASURE} tasks) ---')
sample_bd = TaskBreakdown(
    original_prompt=full_bd.original_prompt,
    tasks=full_bd.tasks[:PIPELINE_MEASURE],
    total_estimated_hours=sum(t.estimated_hours for t in full_bd.tasks[:PIPELINE_MEASURE]),
)
measure_tasks(sample_bd, model=LARGEST)

# Step 3: Cost recommendation
if sample_bd.measurements:
    measured_avg = sum(m.total_tokens for m in sample_bd.measurements) / len(sample_bd.measurements)
    diff_avg = sum(t.difficulty for t in full_bd.tasks) / len(full_bd.tasks)
    _, info, cost, why = select_best_model(PIPELINE_BUDGET, diff_avg, actual_tokens_per_task=measured_avg)
    print('\n=== PIPELINE RECOMMENDATION ===')
    print(f'Model         : {info["display_name"]}')
    print(f'Estimated cost: ${cost:.4f} of ${PIPELINE_BUDGET:.2f}')
    print(f'Why           : {why}')

Project : Build a mobile expense tracker with user login, add/edit/delete expenses, category tags, monthly summary charts, and CSV export
Budget  : $10.0
Model   : mistral:7b

--- Task Breakdown ---
  [estimator] progress 1/1 at 515 tokens → projected ~515 total → stopping stream at 515 (50%)
  [estimator] reached 50% threshold (515 tokens), closing stream
  Token usage: prompt=0, response=515, total=515
Task Breakdown: Build a mobile expense tracker with user login, add/edit/delete expenses, category tags, monthly summary charts, and CSV export
Total Estimated Hours: 54.0
--------------------------------------------------------------------------------

1. Task Title One: Design the Mobile Expense Tracker User Interface (UI)
   Difficulty: 🔴🔴🔴⚪⚪ (3/5)
   Estimated: 6.0 hours
   Description: 
   Dependencies: none

2. Task Title Two: Implement User Authentication System
   Difficulty: 🔴🔴🔴🔴⚪ (4/5)
   Estimated: 10.0 hours
   Description: 
   Dependencies: task title one (ui design)

3. T

## 8 · CLI Tool Demo

Runs `task_breaker.py` as a subprocess, the same way a user would from the terminal.

In [16]:
cli_result = subprocess.run(
    [
        sys.executable,
        os.path.join(SRC, 'tools', 'task_breaker.py'),
        '--prompt', 'Build a simple todo app with user accounts',
        '--model-size', 'largest',
    ],
    capture_output=True,
    text=True,
    cwd=ROOT,
)
print(cli_result.stdout)
if cli_result.returncode != 0:
    print('STDERR:', cli_result.stderr)

Breaking down task using model: mistral:7b
  [estimator] progress 1/9 at 609 tokens → projected ~5481 total → stopping stream at 2740 (50%)
  Token usage: prompt=380, response=5481, total=5861
Task Breakdown: Build a simple todo app with user accounts
Total Estimated Hours: 50.0
--------------------------------------------------------------------------------

1. Task Title One: Design the architecture of the todo app
   Difficulty: 🔴🔴🔴🔴⚪ (4/5)
   Estimated: 6.0 hours
   Description: 
   Dependencies: none

2. Task Title Two: Set up development environment
   Difficulty: 🔴🔴⚪⚪⚪ (2/5)
   Estimated: 2.0 hours
   Description: 
   Dependencies: none

3. Task Title Three: Design user interface (UI) wireframes
   Difficulty: 🔴🔴🔴⚪⚪ (3/5)
   Estimated: 4.0 hours
   Description: 
   Dependencies: task title one

4. Task Title Four: Implement user authentication
   Difficulty: 🔴🔴🔴🔴⚪ (4/5)
   Estimated: 8.0 hours
   Description: 
   Dependencies: tasks title one, three

5. Task Title Five: Create t

In [21]:
# JSON output mode
cli_json = subprocess.run(
    [
        sys.executable,
        os.path.join(SRC, 'tools', 'task_breaker.py'),
        '--prompt', 'Set up CI/CD with GitHub Actions',
        '--model-size', 'largest',
        '--json',
    ],
    capture_output=True,
    text=True,
    cwd=ROOT,
)
if cli_json.returncode == 0:
    # Extract JSON portion (skip any non-JSON lines like token usage)
    lines = cli_json.stdout.strip().splitlines()
    json_start = next((i for i, l in enumerate(lines) if l.strip().startswith('{')), 0)
    data = json.loads('\n'.join(lines[json_start:]))
    print(f'Tasks returned: {len(data["tasks"])}')
    print(f'Total hours   : {data["total_estimated_hours"]}')
    for t in data['tasks'][:3]:
        print(f'  {t["title"]} (diff={t["difficulty"]}, {t["estimated_hours"]}h)')
else:
    print('STDERR:', cli_json.stderr)

Tasks returned: 16
Total hours   : 22.0
  Research and Understand GitHub Actions Workflow (diff=3, 1.0h)
  Familiarize oneself with the concepts, structure, and capabilities of GitHub Actions. (diff=2, 1.0h)
  Create a Basic GitHub Repository (diff=3, 1.0h)


## 9 · HTTP Microservice

Starts `task_breaker_service.py` on port 5001, runs a health check and a task-break request, then stops the service.

In [22]:
SERVICE_SCRIPT = os.path.join(SRC, 'tools', 'task_breaker_service.py')

# Start service
svc = subprocess.Popen(
    [sys.executable, SERVICE_SCRIPT],
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)  # wait for it to bind

# Health check
try:
    with urllib.request.urlopen('http://127.0.0.1:5001/health', timeout=5) as r:
        health = json.loads(r.read().decode())
        print('Health check:', health)
except Exception as e:
    print(f'Service not ready: {e}')

Health check: {'status': 'healthy', 'service': 'task-breaker'}


In [23]:
# Break a task via the HTTP service
payload = json.dumps({
    'task': 'Build a calculator app with basic arithmetic',
    'model': LARGEST,
    'max_retries': 2,
}).encode()

req = urllib.request.Request(
    'http://127.0.0.1:5001/break-task',
    data=payload,
    headers={'Content-Type': 'application/json'},
    method='POST',
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        resp = json.loads(r.read().decode())

    if resp.get('status') == 'success':
        print(f'Tasks   : {resp["task_count"]}')
        print(f'Hours   : {resp["total_estimated_hours"]:.1f}h')
        print()
        for t in resp['tasks']:
            bar = 'X' * t['difficulty'] + '.' * (5 - t['difficulty'])
            print(f'  [{bar}] {t["title"]}')
    else:
        print('Error:', resp.get('error'))
except Exception as e:
    print(f'Request failed: {e}')

Tasks   : 8
Hours   : 31.0h

  [XXX..] Task Title One: Define the structure of the calculator app
  [XXX..] Task Title Two: Implement basic UI elements
  [XXXX.] Task Title Three: Implement basic arithmetic operations (addition, subtraction, multiplication, division)
  [XXX..] Task Title Four: Implement operator priority and order of operations
  [XXX..] Task Title Five: Implement parentheses support
  [XX...] Task Title Six: Implement decimal points and rounding
  [XXX..] Task Title Seven: Implement error handling and validation
  [XX...] Task Title Eight: Test the calculator app


In [24]:
# Stop the service
svc.terminate()
svc.wait(timeout=5)
print('Service stopped.')

Service stopped.


## 10 · Summary

Re-prints the key numbers from the session.

In [25]:
print('=== SESSION SUMMARY ===')
print()
print(f'Smallest model  : {SMALLEST}')
print(f'Largest model   : {LARGEST}')
print()
print(f'Breakdown project      : {PROJECT}')
print(f'Tasks returned         : {len(breakdown.tasks)}')
print(f'Total estimated hours  : {breakdown.total_estimated_hours:.1f}h')
print()
if measurements:
    print(f'Tasks measured         : {len(measurements)}')
    print(f'Avg tokens / task      : {avg_tokens:.0f}')
    print(f'Budget (section 6)     : ${BUDGET}')
    print(f'Recommended model      : {model_info["display_name"]}')
    print(f'Estimated cost         : ${est_cost:.4f}')

=== SESSION SUMMARY ===

Smallest model  : qwen2.5:3b
Largest model   : mistral:7b

Breakdown project      : Build a REST API with JWT authentication and a PostgreSQL database
Tasks returned         : 14
Total estimated hours  : 22.0h

Tasks measured         : 3
Avg tokens / task      : 93
Budget (section 6)     : $5.0
Recommended model      : Llama 2 70B (Premium - ML Ready)
Estimated cost         : $0.0074
